In [25]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
import numpy as np

In [27]:
from bayesgpt.simulators import NestedModelFamily, ModelVariant, Tokenizer
from bayesgpt.simulators.benchmarks import SuperDDM, StandardDDM, CollapsingBoundDDM

### Metas

In [28]:
num_samples = 1000  # Global number of samples per model variant

In [29]:
# Define modulation function for context-dependent parameters in SuperDDM
def modulation(params, context):
    """Adjust drift rate based on context (e.g., stimulus strength)."""
    params = params.copy()
    if "v" in params:
        params["v"] = params["v"] * (1 + context[0])  # Scale drift rate
    elif "v_components" in params:
        params["v_components"] = params["v_components"] * (1 + context[0])
    elif "v_schedule" in params:
        params["v_schedule"] = params["v_schedule"] * (1 + context[0])
    return params

In [30]:
# Common tokenizer parameters
parameter_names = [
    "v",
    "a",
    "z",
    "tau",
    "sigma",
    "angle",
    "s_v",
    "s_z",
    "s_tau",
    "v_components",
    "p_components",
    "v_schedule",
    "t_schedule"
]

### DDM Variants

In [31]:
super_ddm_params = parameter_names

In [32]:
fixed_parameters = {
    "p_components": np.array([0.6, 0.4]),  # Mixture probabilities
    "t_schedule": np.array([0.0, 0.5])  # Time points for scheduled drifts
}
free_parameters = {
    "a": lambda c: np.random.uniform(0.8, 1.2, 1),  # Decision boundary
    "sigma": lambda c: np.random.uniform(0.05, 0.15, 1),  # Diffusion noise
    "s_v": lambda c: np.random.uniform(0.01, 0.1, 1),  # Drift rate noise
    "angle": lambda c: np.random.uniform(0.0, 0.05, 1),  # Boundary collapse
    "s_z": lambda c: np.random.uniform(0.005, 0.02, 1),  # Starting point noise
    "s_tau": lambda c: np.random.uniform(0.005, 0.02, 1),  # Non-decision time noise
    "v_components": lambda c: np.random.randn(2) * 0.5,  # Mixture drift rates
    "v_schedule": lambda c: np.random.randn(2) * 0.5,  # Scheduled drift rates
    "z": lambda c: np.random.uniform(0.4, 0.6, 1),  # Starting point
    "tau": lambda c: np.random.uniform(0.1, 0.3, 1)  # Non-decision time
}
parameter_dims = {
    "a": 1, "v": 1, "sigma": 1, "s_v": 1, "z": 1, "tau": 1, "angle": 1,
    "s_z": 1, "s_tau": 1, "v_components": 2, "p_components": 2,
    "v_schedule": 2, "t_schedule": 2
}

In [33]:
tokenizer_super_mixture = Tokenizer(
    parameter_names=parameter_names,
    variant_parameters=["a", "v_components", "p_components", "sigma", "s_v", "z", "tau", "angle", "s_z", "s_tau"],
    fixed_parameters=fixed_parameters,
    free_parameters=free_parameters,
    parameter_dims=parameter_dims,
    context_shape=(1,)
)
tokenizer_super_schedule = Tokenizer(
    parameter_names=parameter_names,
    variant_parameters=["a", "v_schedule", "t_schedule", "sigma", "s_v", "z", "tau", "angle", "s_z", "s_tau"],
    fixed_parameters=fixed_parameters,
    free_parameters=free_parameters,
    parameter_dims=parameter_dims,
    context_shape=(1,)
)
tokenizer_standard = Tokenizer(
    parameter_names=parameter_names,
    variant_parameters=["v", "a", "sigma", "s_v", "z", "tau", "s_z", "s_tau"],
    fixed_parameters=fixed_parameters,
    free_parameters=free_parameters,
    parameter_dims=parameter_dims,
    context_shape=(1,)
)
tokenizer_collapsing = Tokenizer(
    parameter_names=parameter_names,
    variant_parameters=["v", "a", "sigma", "s_v", "z", "tau", "angle", "s_z", "s_tau"],
    fixed_parameters=fixed_parameters,
    free_parameters=free_parameters,
    parameter_dims=parameter_dims,
    context_shape=(1,)
)

In [34]:
model_variant_mixture = ModelVariant(
    name="super_ddm_mixture",
    model=SuperDDM,
    tokenizer=tokenizer_super_mixture,
    num_samples=num_samples
)
model_variant_schedule = ModelVariant(
    name="super_ddm_schedule",
    model=SuperDDM,
    tokenizer=tokenizer_super_schedule,
    num_samples=num_samples
)
model_variant_standard = ModelVariant(
    name="standard_ddm",
    model=StandardDDM,
    tokenizer=tokenizer_standard,
    num_samples=num_samples
)
model_variant_collapsing = ModelVariant(
    name="collapsing_bound_ddm",
    model=CollapsingBoundDDM,
    tokenizer=tokenizer_collapsing,
    num_samples=num_samples
)

In [35]:
# Cell 3: Run simulations and print results
context = np.array([0.5], dtype=np.float32)  # Single simulation context
result_super_mixture = model_variant_mixture.sample(context=context)
result_super_schedule = model_variant_schedule.sample(context=context)
result_standard = model_variant_standard.sample(context=context)
result_collapsing = model_variant_collapsing.sample(context=context)

In [36]:
def print_variant_results(variant: ModelVariant, result: dict, tokenizer: Tokenizer, num_samples: int):
    """Print simulation results and summary statistics for a ModelVariant."""
    print(f"\n{variant.name} Results:")
    print("Variant Name:", result["variant_name"])
    print("Simulated Data Keys:", result["sim_data"].keys())
    print("Reaction Times (first 5):", result["sim_data"]["rts"][:5])
    print("Choices (first 5):", result["sim_data"]["choices"][:5])
    print("Full Parameters (shape):", result["full_params"].shape)
    print("Inference Conditions (shape):", result["inference_conditions"].shape)

    # Summarize results using the model's summarize method
    summary = variant.model.summarize(
        outputs=result["sim_data"],
        quantile_levels=[0.1, 0.3, 0.5, 0.7, 0.9],
        by_choice=True,
        tau=np.full(num_samples, result["full_params"][
            tokenizer.parameter_slices["tau"]][0], dtype=np.float32)
    )
    print(f"{variant.name} Summary:")
    print("Invalid Rate:", summary["invalid_rate"])
    print("RT Quantiles:", summary["rt_quantiles"])
    print("RT Quantiles by Choice:\n", summary["rt_quantiles_by_choice"])
    print("Decision Time Quantiles:", summary["dt_quantiles"])
    print("Decision Time Quantiles by Choice:\n", summary["dt_quantiles_by_choice"])

In [37]:
print_variant_results(model_variant_mixture, result_super_mixture, tokenizer_super_mixture, num_samples)
print_variant_results(model_variant_schedule, result_super_schedule, tokenizer_super_schedule, num_samples)
print_variant_results(model_variant_standard, result_standard, tokenizer_standard, num_samples)
print_variant_results(model_variant_collapsing, result_collapsing, tokenizer_collapsing, num_samples)


super_ddm_mixture Results:
Variant Name: super_ddm_mixture
Simulated Data Keys: dict_keys(['rts', 'choices', 'context'])
Reaction Times (first 5): [2.481011  4.332432  1.1634243 1.3896449 1.8070489]
Choices (first 5): [1. 1. 1. 1. 1.]
Full Parameters (shape): (17,)
Inference Conditions (shape): (34,)
super_ddm_mixture Summary:
Invalid Rate: 0.11
RT Quantiles: [1.4842322 2.051719  2.680031  3.5562508 6.1935124]
RT Quantiles by Choice:
 [[9.119479  9.241481  9.510396  9.570025  9.873662 ]
 [1.4841331 2.0505223 2.6761067 3.547133  5.985889 ]]
Decision Time Quantiles: [1.3636063 1.9310931 2.5594053 3.435625  6.0728865]
Decision Time Quantiles by Choice:
 [[8.998854  9.120855  9.3897705 9.4494    9.7530365]
 [1.3635073 1.9298965 2.555481  3.4265072 5.865263 ]]

super_ddm_schedule Results:
Variant Name: super_ddm_schedule
Simulated Data Keys: dict_keys(['rts', 'choices', 'context'])
Reaction Times (first 5): [4.2231107 5.2985525 5.8695164 5.165837  4.3614845]
Choices (first 5): [0. 0. 0. 0.

### `NestedModelFamily`

In [40]:
# Initialize NestedModelFamily with all variants
model_family = NestedModelFamily(
    variants=[model_variant_mixture, model_variant_schedule, model_variant_standard, model_variant_collapsing],
    n_jobs=2
)

In [42]:
# Batch sample parameters
batch_params = model_family.batch_sample(
    num_samples_per_variant=num_samples,
    context=context
)

In [43]:
print("\nBatch Sampled Parameters:")
for i, params in enumerate(batch_params["parameters"]):
    print(f"Variant {model_family.variant_names[i]} Parameters (first sample):")
    for key, value in params.items():
        print(f"{key}: {value[:5] if isinstance(value, np.ndarray) else value}")


Batch Sampled Parameters:
Variant super_ddm_mixture Parameters (first sample):
a: [0.8610322]
z: [0.55126196]
sigma: [0.13645762]
p_components: [0.6 0.4]
tau: [0.22198065]
s_tau: [0.00914211]
s_v: [0.05218232]
s_z: [0.01014872]
angle: [0.03930838]
v_components: [0.32767898 0.6170668 ]
Variant super_ddm_schedule Parameters (first sample):
s_v: [0.07717462]
v_schedule: [0.1677061  0.53095055]
a: [1.1388388]
s_tau: [0.0166151]
t_schedule: [0.  0.5]
s_z: [0.01904838]
z: [0.42615426]
sigma: [0.08747325]
angle: [0.00326552]
tau: [0.15119566]
Variant standard_ddm Parameters (first sample):
a: [1.0752722]
z: [0.4567465]
sigma: [0.12414446]
tau: [0.13709214]
s_v: [0.03723646]
s_z: [0.0067093]
s_tau: [0.01639634]
v: [-2.173599]
Variant collapsing_bound_ddm Parameters (first sample):
a: [0.93699735]
z: [0.43876687]
sigma: [0.14934936]
tau: [0.28845754]
s_tau: [0.01893823]
s_v: [0.08093733]
s_z: [0.01825779]
angle: [0.04962123]
v: [-1.1788831]


In [44]:
# Run batch simulations with modulation
batch_results = model_family.batch_simulate(
    num_samples_per_variant=num_samples,
    context=[context] * len(model_family.variants),  # Same context for all variants
    modulation=modulation
)

In [45]:
print("\nBatch Simulation Results:")
print(f"sim_data shape: {batch_results['sim_data'].shape}")  # (4, 1000, 2)
print(f"full_params shape: {batch_results['full_params'].shape}")  # (4, 1000, 13)
print(f"inference_conditions shape: {batch_results['inference_conditions'].shape}")  # (4, 1000, 26)
print(f"variant_names: {batch_results['variant_names']}")
print(f"variant_indices shape: {batch_results['variant_indices'].shape}")  # (4, 1)
print(f"context shape: {batch_results['context'].shape}")  # (4, 1)


Batch Simulation Results:
sim_data shape: (4,)
full_params shape: (4, 17)
inference_conditions shape: (4, 34)
variant_names: ['super_ddm_mixture', 'super_ddm_schedule', 'standard_ddm', 'collapsing_bound_ddm']
variant_indices shape: (4, 1)
context shape: (4, 1)
